In [0]:
import requests

import pandas as pd

from pyspark.sql import SparkSession

from pyspark.sql import functions as F

from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()

# 🔐 GET FROM DATABRICKS SECRETS (FIX)

BASE_URL = dbutils.secrets.get(scope="slainte-secrets", key="supabase-url").strip()

API_KEY  = dbutils.secrets.get(scope="slainte-secrets", key="supabase-api-key").strip()

HEADERS  = {"apikey": API_KEY, "Authorization": f"Bearer {API_KEY}"}

def fetch(table):

    df = pd.DataFrame(requests.get(f"{BASE_URL}/{table}", headers=HEADERS).json())

    return df.where(lambda x: x.notna(), None)

# ----------------------------

# Fetch

# ----------------------------

df_groups = spark.createDataFrame(fetch("sla_groups"))

df_config = spark.createDataFrame(fetch("sla_configurations"))

df_integrations = spark.createDataFrame(fetch("jira_integrations"))

# ----------------------------

# Cast IDs

# ----------------------------

df_groups = df_groups.withColumn(

    "sla_configuration_id", F.col("sla_configuration_id").cast(StringType())

)

df_config = df_config.withColumn("id", F.col("id").cast(StringType())) \
                     .withColumn("integration_id", F.col("integration_id").cast(StringType()))

df_integrations = df_integrations.withColumn("id", F.col("id").cast(StringType()))

# ----------------------------

# Alias

# ----------------------------

g = df_groups.alias("g")

c = df_config.alias("c")

i = df_integrations.alias("i")

# ----------------------------

# Join

# ----------------------------

df = g.join(

    c,

    g["sla_configuration_id"] == c["id"],

    "left"

).join(

    i.select(F.col("id").alias("integration_id"), "user_id"),

    on="integration_id",

    how="left"

)

# ----------------------------

# Select final columns

# ----------------------------

df_final = df.select(

    "g.group_name",

    "g.labels",

    "g.response_time_minutes",

    "g.resolution_time_hours",

    "g.created_at",

    "g.updated_at",

    # project_key from config

    F.col("c.project_key").alias("project_key"),

    "c.project",

    "c.team",

    "c.country",

    "c.calendar_type",

    "c.working_days",

    "c.work_start_time",

    "c.work_end_time",

    "c.timezone",

    "user_id",

    F.current_timestamp().alias("ingestion_timestamp")

)

# ----------------------------

# Write

# ----------------------------

TARGET = "workspace.slainte_bronze.sla_groups_bronze"

df_final.write.mode("overwrite") \
    .format("delta") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET)

print("✅ Saved:", TARGET)

print("Rows:", df_final.count())
 